## GAIA SimCLR

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

c:\Users\Rajarshi\anaconda3\envs\astroml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class GaiaSimCLRDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        feature_cols,
        noise_std=0.05,
        drop_prob=0.1,
        # limit=500
    ):
        # self.df = pd.read_parquet(parquet_path).iloc[:limit]
        self.df = pd.read_parquet(parquet_path)
        self.feature_cols = feature_cols

        # Convert to float32, NaNs allowed for now
        self.X = self.df[feature_cols].astype(np.float32).values

        self.noise_std = noise_std
        self.drop_prob = drop_prob

        # Precompute feature-wise std for scaling noise
        self.feature_std = np.nanstd(self.X, axis=0)
        self.feature_std[self.feature_std == 0] = 1.0

    def __len__(self):
        return len(self.X)

    def _augment(self, x):
        x = x.copy()

        # 1. Gaussian noise (scaled per feature)
        noise = np.random.randn(*x.shape) * self.feature_std * self.noise_std
        x = x + noise

        # 2. Feature dropout
        mask = np.random.rand(*x.shape) < self.drop_prob
        x[mask] = 0.0

        # Replace NaNs after augmentation
        x = np.nan_to_num(x, nan=0.0)

        return x.astype(np.float32)

    def __getitem__(self, idx):
        x = self.X[idx]

        x1 = self._augment(x)
        x2 = self._augment(x)

        return torch.from_numpy(x1), torch.from_numpy(x2)


In [3]:
import torch.nn as nn
import torch.nn.functional as F

In [4]:
class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            # nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            # nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        # Projection head (SimCLR)
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return z


In [5]:
gaia_cols = [
    'gaia_e_RA_ICRS', 'gaia_e_DE_ICRS',
    'gaia_pmRA', 'gaia_e_pmRA',
    'gaia_pmDE', 'gaia_e_pmDE',
    'gaia_Plx', 'gaia_e_Plx'
]
in_dim=len(gaia_cols)
print(f"Input feature dimension: {in_dim}")

Input feature dimension: 8


In [6]:
from train_utils import find_auto_batch_size, nt_xent_loss

def train_simclr(
    parquet_path,
    feature_cols,
    epochs=100,
    batch_size="auto",
    lr= 1e-3, #0.0003,
    device="cuda" if torch.cuda.is_available() else "cpu"
):
    in_dim=len(feature_cols)
    model = MLPEncoder(in_dim=in_dim).to(device)

    if batch_size == "auto":
        batch_size = find_auto_batch_size(model, in_dim, device, start_batch=1024)

    dataset = GaiaSimCLRDataset(parquet_path=parquet_path, feature_cols=feature_cols)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    total_steps = epochs * len(loader)
    # Use one master progress bar for the most accurate timing
    pbar = tqdm(total=total_steps, desc="Training", unit="batch", dynamic_ncols=True)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for x1, x2 in loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1, z2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            current_loss=loss.item()
            epoch_loss += current_loss #epoch_loss instead of total_loss to get avg loss per epoch

            pbar.update(1)
            pbar.set_postfix({
                "epoch": f"{epoch+1}/{epochs}",
                "loss": f"{current_loss:.4f}"
            })

        if len(loader) > 0:
            avg_loss = epoch_loss / len(loader)
            pbar.write(f"Epoch {epoch+1} finished. Avg Loss: {avg_loss:.4f}")
    
    pbar.close()
    return model

In [ ]:
gaia_cols = [
    'gaia_e_RA_ICRS', 'gaia_e_DE_ICRS',
    'gaia_pmRA', 'gaia_e_pmRA',
    'gaia_pmDE', 'gaia_e_pmDE',
    'gaia_Plx', 'gaia_e_Plx'
]

model = train_simclr(
    parquet_path=r"C:\Raj_Stuff\Coding\VScode\Workspaces\Astro-ML\Data\XMM-Gaia-Cross-Matched Data Separate\gaia_raw.parquet",
    feature_cols=gaia_cols,
    # epochs=100,
    # batch_size=256
)
# torch.save(model.state_dict(), "simclr_gaia.pth")

Searching for optimal batch size starting from 1024...
✅ Success! Max tested: 1024. Selected with safety margin: 819


Training:   0%|          | 207/56300 [00:09<39:52, 23.45batch/s, epoch=1/100, loss=5.8415]

## XMM SimCLR

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [ ]:
class XMMSimCLRDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        feature_cols,
        noise_std=0.05,
        drop_prob=0.1
    ):
        self.df = pd.read_parquet(parquet_path)

        # Ensure numeric
        self.df[feature_cols] = self.df[feature_cols].astype(np.float32)

        self.X = self.df[feature_cols].values

        self.feature_cols = feature_cols
        self.noise_std = noise_std
        self.drop_prob = drop_prob

        self.feature_std = np.nanstd(self.X, axis=0)
        self.feature_std[self.feature_std == 0] = 1.0

        # Identify binary columns
        self.binary_mask = np.array([
            col in ['CONFUSED', 'HIGH_BACKGROUND']
            for col in feature_cols
        ])

    def __len__(self):
        return len(self.X)

    def _augment(self, x):
        x = x.copy()

        # Gaussian noise on continuous features
        noise = np.random.randn(*x.shape) * self.feature_std * self.noise_std
        noise[self.binary_mask] = 0.0
        x += noise

        # Feature dropout
        drop_mask = np.random.rand(*x.shape) < self.drop_prob
        x[drop_mask] = 0.0

        # Binary flip (small probability)
        flip_mask = (
            np.random.rand(*x.shape) < 0.02
        ) & self.binary_mask
        x[flip_mask] = 1.0 - x[flip_mask]

        x = np.nan_to_num(x, nan=0.0)
        return x.astype(np.float32)

    def __getitem__(self, idx):
        x = self.X[idx]

        x1 = self._augment(x)
        x2 = self._augment(x)

        return torch.from_numpy(x1), torch.from_numpy(x2)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )

        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)


In [23]:
from train_utils import find_auto_batch_size, nt_xent_loss

def train_xmm_simclr(
    parquet_path,
    feature_cols,
    epochs=100,
    batch_size="auto",
    lr= 1e-3, #0.0003, 
    device="cuda" if torch.cuda.is_available() else "cpu"
):
    in_dim = len(feature_cols)
    model = MLPEncoder(in_dim=in_dim).to(device)
    
    if batch_size == "auto":
        batch_size = find_auto_batch_size(model, in_dim, device, start_batch=1024)

    dataset = XMMSimCLRDataset(parquet_path=parquet_path, feature_cols=feature_cols)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0
    )


    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    total_steps = epochs * len(loader)
    # Use one master progress bar for the most accurate timing
    pbar = tqdm(total=total_steps, desc="Training", unit="batch", dynamic_ncols=True)

    for epoch in range(epochs):
        epoch_loss = 0.0

        for x1, x2 in loader:
            x1 = x1.to(device)
            x2 = x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1, z2)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            current_loss = loss.item()
            epoch_loss += current_loss

            pbar.update(1)
            pbar.set_postfix({
                "epoch": f"{epoch+1}/{epochs}",
                "loss": f"{current_loss:.4f}"
            })

        if len(loader) > 0:
            avg_loss = epoch_loss / len(loader)
            pbar.write(f"Epoch {epoch+1} finished. Avg Loss: {avg_loss:.4f}")
    
    pbar.close()
    return model


In [7]:
xmm_cols = [
    'SC_POSERR',
    'SC_DET_ML',
    'N_DETECTIONS',
    'CONFUSED',
    'HIGH_BACKGROUND',
    'No/Nx'
]
in_dim=len(xmm_cols)
print(f"Input feature dimension: {in_dim}")

Input feature dimension: 6


In [24]:
xmm_cols = [
    'SC_POSERR',
    'SC_DET_ML',
    'N_DETECTIONS',
    'CONFUSED',
    'HIGH_BACKGROUND',
    'No/Nx'
]

xmm_model = train_xmm_simclr(
    parquet_path=r"C:\Raj_Stuff\Coding\VScode\Workspaces\Astro-ML\Data\XMM-Gaia-Cross-Matched Data Separate\xmm_raw.parquet",
    feature_cols=xmm_cols,
    # epochs=100,
    # batch_size=256
)

torch.save(xmm_model.state_dict(), "simclr_xmm.pth")

Searching for optimal batch size starting from 1024...
✅ Success! Max tested: 1024. Selected with safety margin: 819


Training:   1%|          | 100/9900 [00:09<13:10, 12.40batch/s, epoch=2/100, loss=7.4047]

Epoch 1 finished. Avg Loss: 7.4020


Training:   2%|▏         | 199/9900 [00:19<20:19,  7.95batch/s, epoch=3/100, loss=7.3885]

Epoch 2 finished. Avg Loss: 7.3975


...

Epoch 98 finished. Avg Loss: 6.8848


Training:  99%|█████████▉| 9804/9900 [15:09<00:07, 12.14batch/s, epoch=100/100, loss=6.8221]

Epoch 99 finished. Avg Loss: 6.8734


Training: 100%|██████████| 9900/9900 [15:18<00:00, 10.78batch/s, epoch=100/100, loss=6.8627]

Epoch 100 finished. Avg Loss: 6.8406
